In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df_main = pd.read_csv("data/Cassava_Flood_Data_Coded - Sheet1.csv")
df_equip = pd.read_csv("data/Cassava_Flood_Data_Coded - Equipment used.csv")
df_coping = pd.read_csv("data/Cassava_Flood_Data_Coded - Coping and adaptation strategies.csv")
df_interv = pd.read_csv("data/Cassava_Flood_Data_Coded - Effectiveness of flood-management interventions.csv")


In [2]:
for df in [df_main, df_equip, df_coping, df_interv]:
  df.columns = df.columns.str.strip()


def get_num(df, col_names):
  """Safely converts matched columns to numeric, replacing NaNs with 0."""
  for name in col_names:
    if name in df.columns:
      return pd.to_numeric(df[name], errors="coerce").fillna(0)
  return pd.Series(0, index=df.index)

In [3]:
# 2. GROUP LABELS CORRECTION
# 1 = Flood-free, 2 = Flood-affected
df_main["Group_Label"] = df_main["Flood_Exposure"].map(
    {1: "Flood-free", 2: "Flood-affected"}
)

In [4]:
# 3. LAND AREA CONVERSION
# Convert plots to hectares using 15 plots = 1 hectare
plots = get_num(df_main, ["Farm_Size_Plots", "Farm Size Plots"])
df_main["Hectares_Calc"] = np.where(plots > 0, plots / 15.0, np.nan)
# Fallback to recorded hectares if plots missing
df_main["Hectares_Final"] = df_main["Hectares_Calc"].fillna(
    get_num(df_main, ["Farm_Size_in_Hectares", "Farm Size in Hectares"])
)

In [5]:
# 4. PRODUCTION & REVENUE
total_harvest = get_num(df_main, ["Total_Harvest_Kg", "Total Harvest Kg"])
price_per_kg = get_num(df_main, ["Price_Per_Kg_Naira", "Price Per Kg Naira"])

df_main["Yield_Kg_Ha"] = total_harvest / df_main["Hectares_Final"]
df_main["TR"] = total_harvest * price_per_kg
df_main["TR_per_ha"] = df_main["TR"] / df_main["Hectares_Final"]

In [6]:
# 5. COST BUILD (FIXING MISSING/UNDERSTATED ITEMS)
hired_labor = get_num(
    df_main, ["Hired_Labor_Cost_Naira", "Hired Labor Cost Naira"]
)
# Fixed typo: Naria vs Naira
family_labor = get_num(
    df_main,
    [
        "Family_Labour_Cost_Naria",
        "Family_Labour_Cost_Naira",
        "Family Labour Cost Naria",
    ],
)
inputs = get_num(df_main, ["Input_Cost_Naira", "Input Cost Naira"])
transport = get_num(df_main, ["Transport_Cost_Naira", "Transport Cost Naira"])
extra_flood = get_num(
    df_main, ["Extra_Flood_Cost_Naira", "Extra Flood Cost Naira"]
)

# Fetch levies/processing from equipment sheet
other_costs_price = get_num(
    df_equip, ["Other costs incured_price", "Other costs incured price"]
)
df_equip_other = pd.DataFrame({
    "Respondent_ID": df_equip["Respondent_ID"],
    "Other_Costs_Price": other_costs_price,
})
df_main = df_main.merge(df_equip_other, on="Respondent_ID", how="left")
df_main["Other_Costs_Price"] = df_main["Other_Costs_Price"].fillna(0)

# Total Variable Cost (TVC) including extra flood costs
df_main["TVC"] = (
    hired_labor
    + family_labor
    + inputs
    + transport
    + extra_flood
    + df_main["Other_Costs_Price"]
)

In [7]:
# 6. EQUIPMENT DEPRECIATION (MULTIPLIED BY TOOL COUNT)
hoe_num = get_num(df_equip, ["Hoe_num", "Hoe num"])
hoe_dep_unit = get_num(
    df_equip, ["Annual depreciation(hoe)", "Annual depreciation (hoe)"]
)

cutlass_num = get_num(df_equip, ["Cutlass_num", "Cutlass num"])
cutlass_dep_unit = get_num(
    df_equip, ["Annual depreciation(cutlass)", "Annual depreciation (cutlass)"]
)

others_num = get_num(df_equip, ["Number_other", "Number other"])
others_dep_unit = get_num(
    df_equip, ["Annual depreciation(others)", "Annual depreciation (others)"]
)

df_equip["Depreciation_Total"] = (
    (hoe_num * hoe_dep_unit)
    + (cutlass_num * cutlass_dep_unit)
    + (others_num * others_dep_unit)
)

df_main = df_main.merge(
    df_equip[["Respondent_ID", "Depreciation_Total"]],
    on="Respondent_ID",
    how="left",
)
df_main["Depreciation_Total"] = df_main["Depreciation_Total"].fillna(0)

land_rent = get_num(df_main, ["Land_Rent_Naira", "Land Rent Naira"])
df_main["TFC"] = land_rent + df_main["Depreciation_Total"]

In [8]:
# 7. TOTAL COST & PROFIT METRICS
df_main["TC"] = df_main["TVC"] + df_main["TFC"]
df_main["GM"] = df_main["TR"] - df_main["TVC"]
df_main["NFI"] = df_main["TR"] - df_main["TC"]

df_main["TC_per_ha"] = df_main["TC"] / df_main["Hectares_Final"]
df_main["GM_per_ha"] = df_main["GM"] / df_main["Hectares_Final"]
df_main["NFI_per_ha"] = df_main["NFI"] / df_main["Hectares_Final"]

print("Full cost build completed successfully!")

Full cost build completed successfully!


In [9]:
# 8. HYPOTHESIS TESTING (WELCH'S T-TEST: AFFECTED VS FREE)
def run_ttest(var_name, label):
  affected = df_main[df_main["Flood_Exposure"] == 2][var_name].dropna()
  free = df_main[df_main["Flood_Exposure"] == 1][var_name].dropna()

  t_stat, p_val = stats.ttest_ind(affected, free, equal_var=False)
  print(
      f"{label:32} | Affected Mean: {affected.mean():10.2f} | Free Mean:"
      f" {free.mean():10.2f} | t-stat: {t_stat:6.2f} | p-value: {p_val:.4f}"
  )


print("\n--- HYPOTHESIS TESTS (WELCH'S T-TEST) ---")
run_ttest("Yield_Kg_Ha", "Yield (kg/ha)")
run_ttest("TR", "Total Revenue per farm (Naira)")
run_ttest("TR_per_ha", "Total Revenue per ha (Naira)")
run_ttest("NFI", "Net Farm Income per farm (Naira)")
run_ttest("NFI_per_ha", "Net Farm Income per ha (Naira)")


--- HYPOTHESIS TESTS (WELCH'S T-TEST) ---
Yield (kg/ha)                    | Affected Mean:    7876.93 | Free Mean:   14574.36 | t-stat:  -7.15 | p-value: 0.0000
Total Revenue per farm (Naira)   | Affected Mean:  503731.25 | Free Mean:  651336.88 | t-stat:  -1.83 | p-value: 0.0714
Total Revenue per ha (Naira)     | Affected Mean: 1550315.61 | Free Mean: 2692024.22 | t-stat:  -6.18 | p-value: 0.0000
Net Farm Income per farm (Naira) | Affected Mean:  231820.26 | Free Mean:  349232.22 | t-stat:  -1.83 | p-value: 0.0717
Net Farm Income per ha (Naira)   | Affected Mean:  659261.27 | Free Mean: 1444357.54 | t-stat:  -4.40 | p-value: 0.0000
